# Solutions to Exercise 18: Exact Inference in Bayesian Networks

Given the factorised distribution

$P(C,R,D,A,S,F,T) = P(R)P(C)P(D)P(A\vert C)P(F\vert A)P(T\vert F,D) P(S\vert F,T)$

and the univariate distributions given in the code cell below, compute the following quantities using rejection sampling. Verify your answers against the exact solutions computed in Exercise 16.

In [1]:
import numpy as np
PR = np.array([0.7,0.3])
PC = np.array([0.9,0.1])
PD = np.array([0.5,0.5])
PA_C = np.array([[1.0, 0.99],[0.0,0.01]])
PF_A = np.array([[0.999, 0.7],[0.001,0.3]])
PT_FD = np.array([[[0.99,0.40],[0.30,0.01]],[[0.01,0.60],[0.70,0.99]]])
PS_TF = np.array([[[0.999,0.01],[0.70,0.01]],[[0.001,0.99],[0.30,0.99]]])


In [2]:
# First, generate the trace
N = 100000
Trace = np.zeros([N,7])
for i in range(N):
    R = np.random.binomial(1, PR[1])
    C = np.random.binomial(1, PC[1])
    D = np.random.binomial(1, PD[1])
    A = np.random.binomial(1, PA_C[1,C])
    F = np.random.binomial(1, PF_A[1,A])
    T = np.random.binomial(1, PT_FD[1,F,D])
    S = np.random.binomial(1, PS_TF[1,T,F])
    Trace[i:] = np.array([R,C,D,A,F,T,S])
    

1. $P(S)$

In [3]:
# Exact calculation
PA = PA_C @ PC
PF = PF_A @ PA
PT_F = PT_FD @ PD
PT = PT_F @ PF
PS_T = PS_TF@PF
PS = PS_T @ PT

print(f"P(S) = {Trace[:,6].sum()/N} (expect {PS[1]})")


P(S) = 0.09325 (expect 0.09357071278723453)


2. $P(S,C)$

In [4]:
# Exact calculation
PT_A = PT_F @ PF_A
PS_T = PS_TF @ PF
PS_A = PS_T @ PT_A
PS_C = PS_A @ PA_C
PSC = PS_C*PC

# P(S=1,C=1)
CEq1 = Trace[np.where(Trace[:,1]==1)]
SEq1CEq1 = CEq1[np.where(CEq1[:,6]==1)]
PSEq1CEq1 = SEq1CEq1.shape[0]/N
print(f"P(S=1,C=1) = {PSEq1CEq1} (expect {PSC[1,1]})")

P(S=1,C=1) = 0.00953 (expect 0.009400463724620538)


3. $P(A\vert S)$

In [5]:
PSA = PS_A * PA
PA_S = PSA.T/PS
#P(A=0|S=0)
SEq0 = Trace[np.where(Trace[:,6]==0)]
AEq0SEq0 = SEq0[np.where(SEq0[:,3]==0)]
PAEq0GivenSEq0 = AEq0SEq0.shape[0]/SEq0.shape[0]
print(f"P(A=0|S=0) = {PAEq0GivenSEq0} (expect {PA_S[0,0]})")



P(A=0|S=0) = 0.9992390405293631 (expect 0.9990531377523049)


4. $P(A\vert T)$

In [6]:
PTA = PT_A * PA
PA_T = (PTA.T)/PT

TEq1 = Trace[np.where(Trace[:,5]==1)]
AEq0AEq1 = TEq1[np.where(TEq1[:,3]==0)]
PAEq0AEq1 = AEq0AEq1.shape[0]/TEq1.shape[0]
print(f"P(A=0|T=1) = {PAEq0AEq1} (expect {PA_T[0,1]})")


P(A=0|T=1) = 0.9983920191645063 (expect 0.9984723658172912)
